In [2]:
import pandas as pd
import logging
from pathlib import Path

In [3]:
t = pd.read_csv("CVE_2014_to_2025v2.csv")

In [4]:
t.head()

,CVE_ID,Published,Last_Modified,Status,Description,CVSS_BaseScore,CVSS_Vector,CVSS_Version,CVSS_Severity,Weakness_CWE,Affected_CPEs,Vendor,Product,References
0,CVE-2014-0791,2014-01-03T18:54:13.257,2025-04-11T00:51:21.963,Deferred,Integer overflow in the license_read_scope_lis...,6.8,AV:N/AC:M/Au:N/C:P/I:P/A:P,2.0,MEDIUM,CWE-189,cpe:2.3:a:freerdp:freerdp:1.0.0:*:*:*:*:*:*:*;...,freerdp,freerdp,http://advisories.mageia.org/MGASA-2014-0287.h...
1,CVE-2014-0620,2014-01-08T15:30:02.683,2025-04-11T00:51:21.963,Deferred,Multiple cross-site scripting (XSS) vulnerabil...,4.3,AV:N/AC:M/Au:N/C:N/I:P/A:N,2.0,MEDIUM,CWE-79,cpe:2.3:o:technicolor:tc7200_firmware:std6.01....,technicolor,tc7200; tc7200_firmware,http://www.exploit-db.com/exploits/30668; http...
2,CVE-2014-0621,2014-01-08T15:30:02.730,2025-04-11T00:51:21.963,Deferred,Multiple cross-site request forgery (CSRF) vul...,6.8,AV:N/AC:M/Au:N/C:P/I:P/A:P,2.0,MEDIUM,CWE-352,cpe:2.3:o:technicolor:tc7200_firmware:std6.01....,technicolor,tc7200; tc7200_firmware,http://www.exploit-db.com/exploits/30667; http...
3,CVE-2014-1232,2014-01-08T15:30:02.747,2025-04-11T00:51:21.963,Deferred,Cross-site scripting (XSS) vulnerability in th...,4.3,AV:N/AC:M/Au:N/C:N/I:P/A:N,2.0,MEDIUM,CWE-79,cpe:2.3:a:foliovision:foliopress_wysiwyg:*:*:*...,foliovision,foliopress_wysiwyg,http://secunia.com/advisories/56261; http://wo...
4,CVE-2014-0651,2014-01-08T21:55:06.223,2025-04-11T00:51:21.963,Deferred,The administrative interface in Cisco Context ...,4.9,AV:N/AC:M/Au:S/C:P/I:P/A:N,2.0,MEDIUM,CWE-264,cpe:2.3:a:cisco:context_directory_agent:-:*:*:...,cisco,context_directory_agent,http://osvdb.org/101809; http://secunia.com/ad...


In [25]:
all_date = t["Published"]
ans = []
for i in all_date:
    ans.append(i[:10])

In [4]:
# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

class CVEGraphProcessor:
    """Process CVE data into graph nodes and edges"""
    
    def __init__(self, csv_path):
        self.csv_path = csv_path
        self.df = None
        
    def load_data(self):
        """Load and validate CSV data"""
        try:
            self.df = pd.read_csv(self.csv_path)
            logging.info(f"✓ Loaded {len(self.df)} CVE records")
            return True
        except FileNotFoundError:
            logging.error(f"✗ File not found: {self.csv_path}")
            return False
        except Exception as e:
            logging.error(f"✗ Error loading data: {e}")
            return False
    
    @staticmethod
    def parse_cpe_robust(cpe_str):
        """
        Parses CPE strings safely, handling 2.3 and legacy formats.
        Returns: (Full_CPE, Vendor, Product, Version)
        """
        if not isinstance(cpe_str, str) or not cpe_str.strip():
            return None, None, None, None
        
        parts = cpe_str.split(':')
        
        # CPE 2.3 format: cpe:2.3:a:vendor:product:version:...
        if len(parts) >= 6 and parts[1] == '2.3':
            version = parts[5] if parts[5] not in ['*', '-'] else None
            return cpe_str, parts[3], parts[4], version
        
        # Legacy format: cpe:/a:vendor:product:version
        elif len(parts) >= 4 and parts[0] == 'cpe':
            version = parts[4] if len(parts) > 4 and parts[4] not in ['*', '-'] else None
            return cpe_str, parts[2], parts[3], version
        
        logging.warning(f"Unparseable CPE format: {cpe_str}")
        return None, None, None, None
    
    def create_cve_nodes(self, output_path='nodes_cve.csv'):
        """Create CVE node file"""
        df_cve = self.df[['CVE_ID', 'Description', 'CVSS_BaseScore', 'CVSS_Vector']].drop_duplicates()
        df_cve.to_csv(output_path, index=False)
        logging.info(f"✓ Created {len(df_cve)} CVE nodes → {output_path}")
        return df_cve
    
    def create_cve_cwe_edges(self, output_path='edges_cve_cwe.csv'):
        """
        Create CVE to CWE relationship edges
        FIXED: Now properly explodes multiple CWEs separated by semicolons
        """
        df_cve_cwe = self.df[['CVE_ID', 'Weakness_CWE']].copy().dropna()
        
        # EXPLODE multiple CWEs (same pattern as References)
        df_cve_cwe = df_cve_cwe.assign(
            Weakness_CWE=df_cve_cwe['Weakness_CWE'].str.split(r';\s*')
        ).explode('Weakness_CWE')
        
        # Clean empty values
        df_cve_cwe = df_cve_cwe[df_cve_cwe['Weakness_CWE'].str.strip() != '']
        
        # Remove duplicates
        df_cve_cwe = df_cve_cwe.drop_duplicates()
        
        # Add relation type
        df_cve_cwe['Relation'] = 'HAS_WEAKNESS'
        
        df_cve_cwe.to_csv(output_path, index=False)
        logging.info(f"✓ Created {len(df_cve_cwe)} CVE→CWE edges → {output_path}")
        return df_cve_cwe
    
    def process_cpes(self):
        """Process CPE data with robust parsing"""
        df_edges = self.df[['CVE_ID', 'Affected_CPEs']].copy()
        
        # Explode semi-colon separated CPEs
        df_edges = df_edges.assign(
            Affected_CPEs=df_edges['Affected_CPEs'].str.split(r';\s*')
        ).explode('Affected_CPEs')
        
        # Clean empty values
        df_edges = df_edges[
            df_edges['Affected_CPEs'].notna() & 
            (df_edges['Affected_CPEs'].str.strip() != '')
        ]
        
        # Parse CPEs
        parsed_data = df_edges['Affected_CPEs'].apply(
            lambda x: pd.Series(self.parse_cpe_robust(x))
        )
        parsed_data.columns = ['CPE_ID', 'Vendor', 'Product', 'Version']
        
        # Combine
        df_full_map = pd.concat([df_edges[['CVE_ID']], parsed_data], axis=1)
        df_full_map = df_full_map.dropna(subset=['CPE_ID', 'Vendor', 'Product'])
        
        logging.info(f"✓ Parsed {len(df_full_map)} CPE relationships")
        return df_full_map
    
    def create_cve_cpe_edges(self, df_full_map, output_path='edges_cve_cpe.csv'):
        """Create CVE to CPE edges"""
        df_cve_cpe = df_full_map[['CVE_ID', 'CPE_ID']].drop_duplicates()
        df_cve_cpe['Relation'] = 'AFFECTS'
        df_cve_cpe.to_csv(output_path, index=False)
        logging.info(f"✓ Created {len(df_cve_cpe)} CVE→CPE edges → {output_path}")
        return df_cve_cpe
    
    def create_cpe_product_edges(self, df_full_map, output_path='edges_cpe_product.csv'):
        """Create CPE to Product edges with composite key to avoid vendor collisions"""
        # Use composite key: Vendor:Product to uniquely identify products
        df_cpe_prod = df_full_map[['CPE_ID', 'Vendor', 'Product', 'Version']].copy()
        df_cpe_prod['Product_ID'] = df_cpe_prod['Vendor'] + ':' + df_cpe_prod['Product']
        df_cpe_prod = df_cpe_prod[['CPE_ID', 'Product_ID', 'Version']].drop_duplicates()
        df_cpe_prod['Relation'] = 'IS_VERSION_OF'
        df_cpe_prod.to_csv(output_path, index=False)
        logging.info(f"✓ Created {len(df_cpe_prod)} CPE→Product edges → {output_path}")
        return df_cpe_prod
    
    def create_product_vendor_edges(self, df_full_map, output_path='edges_product_vendor.csv'):
        """Create Product to Vendor edges using composite Product_ID"""
        df_prod_vend = df_full_map[['Vendor', 'Product']].copy()
        df_prod_vend['Product_ID'] = df_prod_vend['Vendor'] + ':' + df_prod_vend['Product']
        df_prod_vend = df_prod_vend[['Product_ID', 'Vendor']].drop_duplicates()
        df_prod_vend['Relation'] = 'MADE_BY'
        df_prod_vend.to_csv(output_path, index=False)
        logging.info(f"✓ Created {len(df_prod_vend)} Product→Vendor edges → {output_path}")
        return df_prod_vend
    
    def create_cve_reference_edges(self, output_path='edges_cve_ref.csv'):
        """Create CVE to Reference edges"""
        df_cve_ref = self.df[['CVE_ID', 'References']].copy().dropna()
        
        # Explode references
        df_cve_ref = df_cve_ref.assign(
            References=df_cve_ref['References'].str.split(r';\s*')
        ).explode('References')
        
        # Clean and deduplicate
        df_cve_ref = df_cve_ref[df_cve_ref['References'].str.strip() != '']
        df_cve_ref = df_cve_ref.drop_duplicates()
        df_cve_ref['Relation'] = 'LINKS_TO'
        
        df_cve_ref.to_csv(output_path, index=False)
        logging.info(f"✓ Created {len(df_cve_ref)} CVE→Reference edges → {output_path}")
        return df_cve_ref
    
    def create_node_files(self, df_full_map):
        """
        Create additional node files for vendors, products, CWEs, and references
        FIXED: CWE nodes now properly created from exploded edges
        """
        # Vendor nodes
        df_vendors = df_full_map[['Vendor']].drop_duplicates()
        df_vendors.to_csv('nodes_vendor.csv', index=False)
        logging.info(f"✓ Created {len(df_vendors)} Vendor nodes → nodes_vendor.csv")
        
        # Product nodes with composite ID
        df_products = df_full_map[['Vendor', 'Product']].copy()
        df_products['Product_ID'] = df_products['Vendor'] + ':' + df_products['Product']
        df_products = df_products[['Product_ID', 'Product', 'Vendor']].drop_duplicates()
        df_products.to_csv('nodes_product.csv', index=False)
        logging.info(f"✓ Created {len(df_products)} Product nodes → nodes_product.csv")
        
        # CPE nodes
        df_cpes = df_full_map[['CPE_ID', 'Product', 'Version']].drop_duplicates()
        df_cpes.to_csv('nodes_cpe.csv', index=False)
        logging.info(f"✓ Created {len(df_cpes)} CPE nodes → nodes_cpe.csv")
        
        # CWE nodes (FIXED: Now uses exploded data)
        df_cwes = self.df[['Weakness_CWE']].copy().dropna()
        
        # Explode CWEs (same as edges)
        df_cwes = df_cwes.assign(
            Weakness_CWE=df_cwes['Weakness_CWE'].str.split(r';\s*')
        ).explode('Weakness_CWE')
        
        # Clean and deduplicate
        df_cwes = df_cwes[df_cwes['Weakness_CWE'].str.strip() != '']
        df_cwes = df_cwes.drop_duplicates()
        
        # Rename column
        df_cwes.rename(columns={'Weakness_CWE': 'CWE_ID'}, inplace=True)
        df_cwes.to_csv('nodes_cwe.csv', index=False)
        logging.info(f"✓ Created {len(df_cwes)} CWE nodes → nodes_cwe.csv")
        
        # Reference nodes (URLs)
        df_refs = self.df[['References']].copy().dropna()
        df_refs = df_refs.assign(References=df_refs['References'].str.split(r';\s*')).explode('References')
        df_refs = df_refs[df_refs['References'].str.strip() != ''].drop_duplicates()
        df_refs.rename(columns={'References': 'Reference_URL'}, inplace=True)
        df_refs.to_csv('nodes_reference.csv', index=False)
        logging.info(f"✓ Created {len(df_refs)} Reference nodes → nodes_reference.csv")
    
    def process_all(self):
        """Execute full processing pipeline"""
        if not self.load_data():
            return False
        
        print("\n" + "="*50)
        print("CVE GRAPH PROCESSING PIPELINE")
        print("="*50 + "\n")
        
        # Create nodes
        self.create_cve_nodes()
        
        # Create edges
        self.create_cve_cwe_edges()
        
        # Process CPEs
        df_full_map = self.process_cpes()
        
        # Create hierarchical edges
        self.create_cve_cpe_edges(df_full_map)
        self.create_cpe_product_edges(df_full_map)
        self.create_product_vendor_edges(df_full_map)
        
        # Create reference edges
        self.create_cve_reference_edges()
        
        # Create additional node files
        self.create_node_files(df_full_map)
        
        print("\n" + "="*50)
        print("✅ PROCESSING COMPLETE")
        print("="*50)
        return True

In [5]:
# Usage
if __name__ == "__main__":
    processor = CVEGraphProcessor('CVE_2014_to_2025v2.csv')
    processor.process_all()

INFO: ✓ Loaded 201751 CVE records



CVE GRAPH PROCESSING PIPELINE



INFO: ✓ Created 201751 CVE nodes → nodes_cve.csv
INFO: ✓ Created 232216 CVE→CWE edges → edges_cve_cwe.csv
INFO: ✓ Parsed 1460346 CPE relationships
INFO: ✓ Created 1374447 CVE→CPE edges → edges_cve_cpe.csv
INFO: ✓ Created 220433 CPE→Product edges → edges_cpe_product.csv
INFO: ✓ Created 89008 Product→Vendor edges → edges_product_vendor.csv
INFO: ✓ Created 555033 CVE→Reference edges → edges_cve_ref.csv
INFO: ✓ Created 22616 Vendor nodes → nodes_vendor.csv
INFO: ✓ Created 89008 Product nodes → nodes_product.csv
INFO: ✓ Created 220433 CPE nodes → nodes_cpe.csv
INFO: ✓ Created 707 CWE nodes → nodes_cwe.csv
INFO: ✓ Created 338646 Reference nodes → nodes_reference.csv



✅ PROCESSING COMPLETE


In [1]:
import pandas as pd

edges_cpe = pd.read_csv("edges_cve_cpe.csv")
edges_cwe = pd.read_csv("edges_cve_cwe.csv")

cpe_counts = edges_cpe.groupby("CPE_ID")["CVE_ID"].count()
print(cpe_counts.describe())
print(f"\nCPEs with >  10 CVEs : {(cpe_counts > 10).sum():,}")
print(f"CPEs with >  50 CVEs : {(cpe_counts > 50).sum():,}")
print(f"CPEs with > 100 CVEs : {(cpe_counts > 100).sum():,}")
print(f"CPEs with > 500 CVEs : {(cpe_counts > 500).sum():,}")

cwe_counts = edges_cwe.groupby("Weakness_CWE")["CVE_ID"].count()
print(f"\nCWE max  : {cwe_counts.max()}")
print(f"CWE mean : {cwe_counts.mean():.1f}")

count    220433.000000
mean          6.235214
std          40.824148
min           1.000000
25%           1.000000
50%           1.000000
75%           4.000000
max        6168.000000
Name: CVE_ID, dtype: float64

CPEs with >  10 CVEs : 20,459
CPEs with >  50 CVEs : 3,339
CPEs with > 100 CVEs : 1,388
CPEs with > 500 CVEs : 179

CWE max  : 27679
CWE mean : 328.5


In [2]:
# See what the mega-CPEs actually are
cpe_counts_named = edges_cpe.groupby("CPE_ID")["CVE_ID"].count()
top_cpes = cpe_counts_named.nlargest(20)
print(top_cpes)

CPE_ID
cpe:2.3:o:linux:linux_kernel:*:*:*:*:*:*:*:*                    6168
cpe:2.3:o:debian:debian_linux:9.0:*:*:*:*:*:*:*                 3859
cpe:2.3:o:microsoft:windows_server_2012:r2:*:*:*:*:*:*:*        3318
cpe:2.3:o:debian:debian_linux:8.0:*:*:*:*:*:*:*                 3288
cpe:2.3:o:apple:iphone_os:*:*:*:*:*:*:*:*                       3274
cpe:2.3:o:debian:debian_linux:10.0:*:*:*:*:*:*:*                3180
cpe:2.3:o:microsoft:windows_server_2012:-:*:*:*:*:*:*:*         3104
cpe:2.3:a:google:chrome:*:*:*:*:*:*:*:*                         2711
cpe:2.3:o:microsoft:windows_server_2019:-:*:*:*:*:*:*:*         2707
cpe:2.3:o:microsoft:windows_server_2016:-:*:*:*:*:*:*:*         2648
cpe:2.3:o:microsoft:windows_server_2008:r2:sp1:*:*:*:*:x64:*    2286
cpe:2.3:o:apple:macos:*:*:*:*:*:*:*:*                           2190
cpe:2.3:o:apple:mac_os_x:*:*:*:*:*:*:*:*                        2139
cpe:2.3:o:google:android:11.0:*:*:*:*:*:*:*                     1958
cpe:2.3:o:microsoft:windows

In [2]:
!pip install matplotlib


  Using cached matplotlib-3.10.9-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (117 kB)
  Using cached kiwisolver-1.5.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
Using cached matplotlib-3.10.9-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (8.8 MB)
Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (362 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.62.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (5.0 MB)
Using cached kiwisolver-1.5.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (1.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [matpl

In [9]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

edges_cwe = pd.read_csv("edges_cve_cwe.csv")
edges_cpe = pd.read_csv("edges_cve_cpe.csv")
edges_prod = pd.read_csv("edges_cpe_product.csv")

# ── CWE degree distribution ──────────────────────────────────
cwe_counts = edges_cwe.groupby("Weakness_CWE")["CVE_ID"].count().sort_values(ascending=False)

print("CWE DEGREE DISTRIBUTION")
print(f"  Total unique CWEs        : {len(cwe_counts):,}")
print(f"  Max CVEs per CWE         : {cwe_counts.max():,}")
print(f"  Median CVEs per CWE      : {cwe_counts.median():.1f}")
print(f"  CWEs with > 1000 CVEs    : {(cwe_counts > 1000).sum():,}")
print(f"  CWEs with > 500  CVEs    : {(cwe_counts > 500).sum():,}")
print(f"  CWEs with > 100  CVEs    : {(cwe_counts > 100).sum():,}")
print(f"  CWEs with > 50   CVEs    : {(cwe_counts > 50).sum():,}")
print(f"  CWEs with <= 10  CVEs    : {(cwe_counts <= 10).sum():,}")
print(f"\nTop 15 CWEs:")
print(cwe_counts.head(15).to_string())

# ── Projected edge count estimates ──────────────────────────
def projected_edges(counts, cap=None):
    """Estimate CVE-CVE edges from clique projection with optional cap"""
    total = 0
    for n in counts:
        effective_n = min(n, cap) if cap else n
        total += (effective_n * (effective_n - 1)) // 2
    return total

print(f"\nCWE PROJECTED EDGE ESTIMATES")
print(f"  Full clique (no cap)     : {projected_edges(cwe_counts):>20,.0f}")
print(f"  With K-sample K=100       : {201751 * 100 // 2:>20,.0f}  (upper bound)")
print(f"  With K-sample K=90       : {201751 * 90 // 2:>20,.0f}  (upper bound)")
print(f"  With K-sample K=80       : {201751 * 80 // 2:>20,.0f}  (upper bound)")
print(f"  With K-sample K=70       : {201751 * 70 // 2:>20,.0f}  (upper bound)")
print(f"  With K-sample K=50       : {201751 * 50 // 2:>20,.0f}  (upper bound)")
print(f"  With K-sample K=20       : {201751 * 20 // 2:>20,.0f}  (upper bound)")
print(f"  With K-sample K=10       : {201751 * 10 // 2:>20,.0f}  (upper bound)")

# ── CVE coverage — how many CVEs have at least one CWE ──────
cves_with_cwe = edges_cwe["CVE_ID"].nunique()
print(f"\nCWE COVERAGE")
print(f"  CVEs with at least 1 CWE : {cves_with_cwe:,} "
      f"({cves_with_cwe/201751*100:.1f}%)")
print(f"  CVEs with no CWE         : {201751 - cves_with_cwe:,} "
      f"({(201751-cves_with_cwe)/201751*100:.1f}%)")

# ── Product degree after wildcard filter ─────────────────────
def is_wildcard_cpe(cpe_str):
    if not isinstance(cpe_str, str):
        return True
    parts = cpe_str.split(":")
    return len(parts) >= 6 and parts[5] == "*"

edges_cpe["is_wildcard"] = edges_cpe["CPE_ID"].apply(is_wildcard_cpe)
edges_cpe_specific = edges_cpe[~edges_cpe["is_wildcard"]]

cpe_to_product = edges_cpe_specific.merge(
    edges_prod[["CPE_ID", "Product_ID"]], on="CPE_ID", how="left"
).dropna(subset=["Product_ID"])

cve_product_map = cpe_to_product[["CVE_ID","Product_ID"]].drop_duplicates()
product_counts  = cve_product_map.groupby("Product_ID")["CVE_ID"].count()

print(f"\nPRODUCT DEGREE DISTRIBUTION (after wildcard CPE filter)")
print(f"  Unique products          : {len(product_counts):,}")
print(f"  Max CVEs per product     : {product_counts.max():,}")
print(f"  Median CVEs per product  : {product_counts.median():.1f}")
print(f"  Products with > 20 CVEs  : {(product_counts > 20).sum():,}")
print(f"  Products with 2-20 CVEs  : {((product_counts>=2)&(product_counts<=20)).sum():,}")
print(f"\nTop 15 Products:")
print(product_counts.nlargest(15).to_string())

CWE DEGREE DISTRIBUTION
  Total unique CWEs        : 707
  Max CVEs per CWE         : 27,679
  Median CVEs per CWE      : 8.0
  CWEs with > 1000 CVEs    : 43
  CWEs with > 500  CVEs    : 59
  CWEs with > 100  CVEs    : 133
  CWEs with > 50   CVEs    : 179
  CWEs with <= 10  CVEs    : 384

Top 15 CWEs:
Weakness_CWE
CWE-79            27679
NVD-CWE-noinfo    27486
CWE-787           12335
CWE-89            11238
CWE-119            8215
CWE-20             8159
CWE-125            7283
CWE-200            6953
CWE-352            5707
CWE-416            5507
CWE-22             5200
NVD-CWE-Other      5194
CWE-78             4017
CWE-476            3665
CWE-862            3471

CWE PROJECTED EDGE ESTIMATES
  Full clique (no cap)     :        1,158,755,577
  With K-sample K=100       :           10,087,550  (upper bound)
  With K-sample K=90       :            9,078,795  (upper bound)
  With K-sample K=80       :            8,070,040  (upper bound)
  With K-sample K=70       :            7,061,28